# NB03 — Geographic holdout evaluation

**Goal:** Test H4 — that the M1 model generalises to an unseen geographic region.

**H4:** Holdout RMSE ≤ 1.25 × in-distribution block CV RMSE (M1 model).

**Held-out region definition:** Australia (latitude −10° to −45°, longitude 110° to 155°). This region is large, reasonably sampled by SPIRE MAGs, and geologically distinct. The definition is set here before seeing the results.

**Design note:** Train on all non-Australia MAGs; predict on Australia. This is a strict test of geographic transfer — the model sees no southern hemisphere ENVO context during training.

**Output:** `data/holdout_results.json`, `figures/nb03_holdout_map.png`.

In [1]:
print("NB03 executing — geographic holdout evaluation (H4).")

NB03 executing — geographic holdout evaluation (H4).


In [2]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from modelling import MAG_DENSITY_FEATURES, geographic_holdout_eval
from evaluation import holdout_vs_cv_ratio

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save
apply_style()

DATA_DIR = Path.cwd().parent / 'data'
FIG_DIR = Path.cwd().parent / 'figures'
FIG_DIR.mkdir(exist_ok=True)

In [3]:
df = pd.read_parquet(DATA_DIR / 'mag_feature_matrix.parquet')
TARGET = 'PF1_Cu'

# Reconstruct full MAG density feature list (primary + subcategory cols)
subcat_density_cols = [c for c in df.columns if c.startswith('ko_per_mb_') and c != 'ko_per_mb_primary']
MAG_ALL_DENSITY = MAG_DENSITY_FEATURES + subcat_density_cols

# Load in-distribution CV RMSE for M1 from NB02
cv_results = pd.read_csv(DATA_DIR / 'cv_results.csv')
m1_cv_rmse = cv_results[cv_results['model'] == 'M1']['rmse'].mean()
print(f"M1 in-distribution CV RMSE: {m1_cv_rmse:.4f}")
print(f"MAG density features for holdout: {MAG_ALL_DENSITY}")

M1 in-distribution CV RMSE: 0.0527
MAG density features for holdout: ['ko_per_mb_primary', 'ko_per_mb_resistance', 'ko_per_mb_transport', 'ko_per_mb_sensing', 'ko_per_mb_metabolism', 'ko_per_mb_cofactor']


In [4]:
# Holdout region: Australia
AUSTRALIA_LAT = (-45, -10)
AUSTRALIA_LON = (110, 155)

holdout_mask = (
    (df['latitude'] >= AUSTRALIA_LAT[0]) & (df['latitude'] <= AUSTRALIA_LAT[1]) &
    (df['longitude'] >= AUSTRALIA_LON[0]) & (df['longitude'] <= AUSTRALIA_LON[1])
).values

# Require complete feature/target rows (MAG density features + target)
required_cols = [TARGET] + MAG_ALL_DENSITY
valid_mask = df[[c for c in required_cols if c in df.columns]].notna().all(axis=1).values

holdout_mask_valid = holdout_mask & valid_mask
print(f"Australia MAGs (valid): {holdout_mask_valid.sum()}")
print(f"Training MAGs (valid): {(~holdout_mask & valid_mask).sum()}")

Australia MAGs (valid): 43
Training MAGs (valid): 15325


In [5]:
if holdout_mask_valid.sum() < 20:
    print("WARNING: fewer than 20 Australia MAGs with valid features — H4 is underpowered.")

holdout_result = geographic_holdout_eval(
    df=df[valid_mask].reset_index(drop=True),
    holdout_mask=holdout_mask[valid_mask],
    feature_cols=MAG_ALL_DENSITY,
    target_col=TARGET,
)
print("Holdout results:", holdout_result)

ratio = holdout_vs_cv_ratio(holdout_result['rmse_holdout'], m1_cv_rmse)
print(f"Holdout/CV RMSE ratio: {ratio:.3f}")
print("H4:", "SUPPORTED (ratio ≤ 1.25)" if ratio <= 1.25 else "NOT SUPPORTED (ratio > 1.25)")

holdout_result['cv_rmse_m1'] = m1_cv_rmse
holdout_result['holdout_cv_ratio'] = ratio
holdout_result['h4_supported'] = bool(ratio <= 1.25)
holdout_result['holdout_region'] = 'Australia'

with open(DATA_DIR / 'holdout_results.json', 'w') as f:
    json.dump(holdout_result, f, indent=2)

Holdout results: {'rmse_holdout': 0.027156755012173166, 'r2_holdout': -0.391694263022732, 'n_holdout': 43, 'n_train': 15325}
Holdout/CV RMSE ratio: 0.515
H4: SUPPORTED (ratio ≤ 1.25)


In [6]:
fig, ax = plt.subplots(figsize=(10, 5))
train_mask = ~holdout_mask & valid_mask
ax.scatter(
    df.loc[train_mask, 'longitude'], df.loc[train_mask, 'latitude'],
    s=5, c='steelblue', alpha=0.4, label='Training MAGs'
)
ax.scatter(
    df.loc[holdout_mask_valid, 'longitude'], df.loc[holdout_mask_valid, 'latitude'],
    s=10, c='#d62728', alpha=0.7, label='Holdout (Australia)'
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geographic holdout — Australia')
ax.legend()
plt.tight_layout()
save(fig, FIG_DIR / 'nb03_holdout_map')